In [1]:
!git clone "https://github.com/patrick-kidger/NeuralCDE.git"

Cloning into 'NeuralCDE'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (41/41), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 253 (delta 38), reused 35 (delta 35), pack-reused 212 (from 1)
Receiving objects: 100% (253/253), 575.32 KiB | 6.54 MiB/s, done.
Resolving deltas: 100% (136/136), done.


In [1]:
!pip install sktime==0.3.1
!pip install torch==1.3.1 torchaudio==0.3.2
!pip install torchdiffeq==0.0.1
!pip install scikit-learn==0.22.1
!pip install tqdm==4.42.1

ERROR: Ignored the following yanked versions: 0.19.0
ERROR: Ignored the following versions that require a different python version: 0.10.0 Requires-Python >=3.7,<3.10; 0.10.1 Requires-Python >=3.7,<3.10; 0.11.0 Requires-Python >=3.7,<3.10; 0.11.1 Requires-Python >=3.7,<3.10; 0.11.2 Requires-Python >=3.7,<3.10; 0.11.3 Requires-Python >=3.7,<3.10; 0.11.4 Requires-Python >=3.7,<3.10; 0.12.0 Requires-Python >=3.7,<3.10; 0.12.1 Requires-Python >=3.7,<3.10; 0.13.0 Requires-Python >=3.7,<3.11; 0.13.1 Requires-Python >=3.7,<3.11; 0.13.2 Requires-Python >=3.7,<3.11; 0.13.3 Requires-Python >=3.7,<3.11; 0.13.4 Requires-Python >=3.7,<3.11; 0.14.0 Requires-Python >=3.7,<3.11; 0.14.1 Requires-Python <3.11,>=3.7; 0.15.0 Requires-Python <3.11,>=3.7; 0.15.1 Requires-Python <3.11,>=3.7; 0.16.0 Requires-Python <3.11,>=3.7
ERROR: Could not find a version that satisfies the requirement sktime==0.3.1 (from versions: 0.1.dev0, 0.1.0, 0.2.0, 0.3.0, 0.4.0, 0.4.1, 0.4.2, 0.4.3, 0.5.2, 0.5.3, 0.6.0, 0.6.1, 0.7.0

In [2]:
import sys
sys.path.append('/kaggle/working/NeuralCDE')
sys.path.append('/kaggle/working/NeuralCDE/experiments')
sys.path.append('/kaggle/working/NeuralCDE/experiments/datasets')
sys.path.append('/kaggle/working/NeuralCDE/experiments/common')

In [3]:
!pip install torchdiffeq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 96.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 76.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 82.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5

In [6]:
!pip install sktime

In [7]:
from experiments.datasets.speech_commands import get_data
from experiments.common import make_model
from experiments.common import main

In [8]:
def main_f(device='cuda', max_epochs=200, *,                                        # training parameters
         model_name, hidden_channels, hidden_hidden_channels, num_hidden_layers,  # model parameters
         dry_run=False,
         **kwargs):                                                               # kwargs passed on to cdeint

    batch_size = 1024
    lr = 0.00005 * (batch_size / 32)

    intensity_data = True if model_name in ('odernn', 'dt', 'decay') else False
    times, train_dataloader, val_dataloader, test_dataloader = get_data(intensity_data,
                                                                                                 batch_size)
    input_channels = 1 + (1 + intensity_data) * 20

    make_model_1 = make_model(model_name, input_channels, 10, hidden_channels, hidden_hidden_channels,
                                   num_hidden_layers, use_intensity=False, initial=True)

    def new_make_model():
        model, regularise = make_model_1()
        model.linear.weight.register_hook(lambda grad: 100 * grad)
        model.linear.bias.register_hook(lambda grad: 100 * grad)
        return model, regularise

    name = None if dry_run else 'speech_commands'
    num_classes = 10
    return main(name, times, train_dataloader, val_dataloader, test_dataloader, device, new_make_model,
                       num_classes, max_epochs, lr, kwargs, step_mode=True)

## Устанавливаю правильную версию torchaudio:

In [9]:
# Устанавливаем современные версии
!pip install torch==1.13.0 torchaudio==0.13.0 torchvision==0.14.0
!pip install torchdiffeq scikit-learn tqdm sktime

import torch
import torchaudio
print(f"Torch: {torch.__version__}, Torchaudio: {torchaudio.__version__}")

# Создаем совместимость для load_wav
if not hasattr(torchaudio, 'load_wav'):
    torchaudio.load_wav = torchaudio.load
    print("Создан алиас: torchaudio.load_wav = torchaudio.load")

ERROR: Ignored the following yanked versions: 2.0.0
ERROR: Could not find a version that satisfies the requirement torchaudio==0.13.0 (from versions: 2.0.1, 2.0.2, 2.1.0, 2.1.1, 2.1.2, 2.2.0, 2.2.1, 2.2.2, 2.3.0, 2.3.1, 2.4.0, 2.4.1, 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0)
ERROR: No matching distribution found for torchaudio==0.13.0
Torch: 2.6.0+cu124, Torchaudio: 2.6.0+cu124
Создан алиас: torchaudio.load_wav = torchaudio.load


In [10]:
import os

# Найдем все файлы с аргументом normalization
project_root = '/kaggle/working/NeuralCDE'

for root, dirs, files in os.walk(project_root):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                if 'normalization' in content and 'load' in content:
                    print(f"Найден файл с normalization: {filepath}")
                    lines = content.split('\n')
                    for i, line in enumerate(lines):
                        if 'normalization' in line and 'load' in line:
                            print(f"  Строка {i+1}: {line.strip()}")
            except Exception as e:
                print(f"Ошибка при чтении {filepath}: {e}")

Найден файл с normalization: /kaggle/working/NeuralCDE/experiments/datasets/speech_commands.py


In [11]:
# Исправляем вызовы с normalization
for root, dirs, files in os.walk(project_root):
    for file in files:
        if file.endswith('.py'):
            filepath = os.path.join(root, file)
            try:
                with open(filepath, 'r', encoding='utf-8') as f:
                    content = f.read()
                
                # Исправляем вызовы с normalization
                # В новых версиях normalization заменяется на нормализацию после загрузки
                new_content = content
                
                # Паттерн: load(..., normalization=True/False, ...)
                import re
                
                # Удаляем аргумент normalization из вызовов load
                new_content = re.sub(
                    r'load\(([^)]*?),\s*normalization\s*=\s*(True|False)([^)]*)\)',
                    r'load(\1\3)',
                    new_content
                )
                
                # Удаляем аргумент normalization из вызовов load_wav  
                new_content = re.sub(
                    r'load_wav\(([^)]*?),\s*normalization\s*=\s*(True|False)([^)]*)\)',
                    r'load(\1\3)',
                    new_content
                )
                
                if content != new_content:
                    with open(filepath, 'w', encoding='utf-8') as f:
                        f.write(new_content)
                    print(f"Исправлен normalization в: {filepath}")
                    
            except Exception as e:
                print(f"Ошибка при обработке {filepath}: {e}")

Исправлен normalization в: /kaggle/working/NeuralCDE/experiments/datasets/speech_commands.py


In [12]:
import torchaudio
import torch

# Создаем совместимую функцию загрузки
def compatible_load(filepath, normalization=False, **kwargs):
    """
    Совместимая функция загрузки аудио
    """
    waveform, sample_rate = torchaudio.load(filepath, **kwargs)
    
    # Применяем normalization если нужно
    if normalization:
        waveform = waveform / (waveform.abs().max() + 1e-8)
    
    return waveform, sample_rate

# Заменяем оригинальные функции
torchaudio.load_wav = compatible_load

# Также патчим обычный load для обработки normalization
original_load = torchaudio.load

def patched_load(filepath, **kwargs):
    if 'normalization' in kwargs:
        normalization = kwargs.pop('normalization')
        waveform, sample_rate = original_load(filepath, **kwargs)
        if normalization:
            waveform = waveform / (waveform.abs().max() + 1e-8)
        return waveform, sample_rate
    else:
        return original_load(filepath, **kwargs)

torchaudio.load = patched_load
print("Функции загрузки успешно пропатчены!")

Функции загрузки успешно пропатчены!


## Запуск кода:

In [13]:
result = main_f(model_name='ncde', hidden_channels=90, 
                              hidden_hidden_channels=40, num_hidden_layers=4)
print(result.keys())  # things we can inspect
print(result.test_metrics.accuracy)

/usr/local/lib/python3.11/dist-packages/torchaudio/functional/functional.py:584: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (64) may be set too high. Or, the value for `n_freqs` (101) may be set too low.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/cuda/memory.py:391: FutureWarning: torch.cuda.reset_max_memory_allocated now calls torch.cuda.reset_peak_memory_stats, which resets /all/ peak memory stats.
  warnings.warn(
  0%|          | 0/200 [00:00<?, ?it/s]

Starting training for model:

NeuralCDE(
  input_channels=21, hidden_channels=90, output_channels=10, initial=True
  (func): FinalTanh(
    input_channels: 21, hidden_channels: 90, hidden_hidden_channels: 40, num_hidden_layers: 4
    (linear_in): Linear(in_features=90, out_features=40, bias=True)
    (linears): ModuleList(
      (0-2): 3 x Linear(in_features=40, out_features=40, bias=True)
    )
    (linear_out): Linear(in_features=40, out_features=1890, bias=True)
  )
  (initial_network): Linear(in_features=21, out_features=90, bias=True)
  (linear): Linear(in_features=90, out_features=10, bias=True)
)




  0%|          | 1/200 [00:57<3:10:32, 57.45s/it]

Epoch: 0  Train loss: 3.82  Train accuracy: 0.104  Val loss: 3.83  Val accuracy: 0.101


  6%|▌         | 11/200 [07:53<2:20:48, 44.70s/it]

...

 56%|█████▌    | 111/200 [1:16:48<1:06:04, 44.55s/it]

Epoch: 110  Train loss: 3.44  Train accuracy: 0.258  Val loss: 3.47  Val accuracy: 0.27


 60%|██████    | 121/200 [1:23:42<54:38, 41.51s/it]  

Epoch: 120  Train loss: 3.44  Train accuracy: 0.258  Val loss: 3.48  Val accuracy: 0.268
Breaking because of no improvement in training loss for 100 epochs.


dict_keys(['times', 'memory_usage', 'baseline_memory', 'num_classes', 'train_dataloader', 'val_dataloader', 'test_dataloader', 'model', 'parameters', 'history', 'train_metrics', 'val_metrics', 'test_metrics'])
0.2593750059604645
